# Trabajo Práctico 2: Armado de un esquema de aprendizaje automático

En el Trabajo Práctico final se espera que puedan poner en práctica los conocimientos adquiridos en el curso, trabajando con un conjunto de datos de clasificación.

El objetivo es que se introduzcan en el desarrollo de un esquema para hacer tareas de aprendizaje automático: selección de un modelo, ajuste de hiperparámetros y evaluación.

El conjunto de datos a utilizar está en `./data/loan_data.csv`.   
El conjunto de datos a utilizar está en `("https://raw.githubusercontent.com/DiploDatos/IntroduccionAprendizajeAutomatico/master/data/loan_data.csv", comment="#")`.

Si abren el archivo verán que al principio (las líneas que empiezan con `#`) describen el conjunto de datos y sus atributos (incluyendo el atributo de etiqueta o clase).

Se espera que hagan uso de las herramientas vistas en el curso. Se espera que hagan uso especialmente de las herramientas brindadas por `scikit-learn`.

# Orientación general del Trabajo Práctico 2

En este trabajo práctico vamos a resolver un problema de **clasificación supervisada**.

El objetivo no es solamente entrenar modelos, sino recorrer el flujo completo de trabajo:

1. Comprender el problema y el dataset.
2. Identificar la variable objetivo.
3. Analizar si las clases están balanceadas.
4. Separar datos de entrenamiento y evaluación.
5. Entrenar modelos de clasificación.
6. Ajustar hiperparámetros con validación cruzada.
7. Evaluar los modelos con métricas apropiadas.
8. Comparar modelos y justificar una recomendación final.

A lo largo del trabajo vamos a intentar responder una pregunta central:

> ¿Qué modelo recomendaríamos usar para este problema y con qué evidencia lo justificaríamos?


# Evaluación con métricas

Para cada modelo evaluado deberíamos reportar, como mínimo:

- accuracy;
- precision;
- recall;
- F1-score;
- matriz de confusión.

Pero además debemos interpretar esos números.

Preguntas orientadoras:

1. ¿El modelo clasifica igual de bien ambas clases?
2. ¿Hay muchos falsos positivos?
3. ¿Hay muchos falsos negativos?
4. ¿Qué métrica parece más relevante para este problema?
5. ¿La accuracy alcanza para decidir o necesitamos mirar otras métricas?

Recordemos que, en problemas con clases desbalanceadas, la accuracy puede ocultar errores importantes.


# Análisis exploratorio

Antes de entrenar modelos, revisemos:

1. Tamaño del dataset.
2. Nombre y tipo de las variables.
3. Valores faltantes.
4. Distribución de la variable `TARGET`.
5. Posible desbalance de clases.

En particular, para `TARGET` deberíamos calcular:

```python
df["TARGET"].value_counts()
df["TARGET"].value_counts(normalize=True)
```

Si una clase aparece mucho más que la otra, la accuracy puede ser engañosa. En ese caso, deberemos mirar también precision, recall, F1-score y matriz de confusión.


## Antes de empezar: costo del error

Como el problema está relacionado con la aprobación de préstamos o créditos, no todos los errores tienen el mismo significado.

Conviene pensar desde el comienzo:

- **Falso positivo:** el modelo recomienda aprobar un préstamo que no debería aprobarse.
- **Falso negativo:** el modelo recomienda rechazar un préstamo que sí debería aprobarse.

Preguntas para tener presentes durante todo el trabajo:

- ¿Cuál de estos errores sería más costoso para el banco?
- ¿Cuál sería más perjudicial para el cliente?
- ¿Qué métrica nos ayudaría a controlar mejor cada tipo de error?


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 200)


## Recomendación sobre la partición Train/Test

Como estamos trabajando con un problema de clasificación, conviene que la proporción de clases sea parecida en entrenamiento y test.

Para eso podemos usar:

```python
train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

El argumento `stratify=y` ayuda a conservar la proporción de clases en ambas particiones.


## Carga de datos y división en entrenamiento y evaluación

La celda siguiente se encarga de la carga de datos (haciendo uso de pandas). Estos serán los que se trabajarán en el resto del laboratorio.

El archivo corresponde a datos de préstamos con garantía hipotecaria. La variable `TARGET` indica si el cliente terminó incumpliendo el préstamo o tuvo mora grave.

- `TARGET = 0`: préstamo pagado.
- `TARGET = 1`: cliente c

In [ ]:
dataset = pd.read_csv("https://raw.githubusercontent.com/DiploDatos/IntroduccionAprendizajeAutomatico/master/data/loan_data.csv", comment="#")
dataset.head()

,TARGET,LOAN,MORTDUE,VALUE,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC
0,0,4700,88026.0000,115506.0000,6.0000,0.0000,0.0000,182.2483,0.0000,27.0000,29.2090
1,0,19300,39926.0000,101208.0000,4.0000,0.0000,0.0000,140.0516,0.0000,14.0000,31.5457
2,0,5700,71556.0000,79538.0000,2.0000,0.0000,0.0000,92.6431,0.0000,15.0000,41.2100
3,0,13000,44875.0000,57713.0000,0.0000,1.0000,0.0000,184.9903,1.0000,12.0000,28.6021
4,0,19300,72752.0000,106084.0000,11.0000,0.0000,0.0000,193.7071,1.0000,13.0000,30.6861


## Análisis exploratorio

Revisamos tamaño, tipos de variables, valores faltantes y distribución de la variable objetivo.

In [ ]:
print("Tamaño del dataset:", dataset.shape)
print()
print("Tipos de variables:")
print(dataset.dtypes)

Tamaño del dataset: (1854, 11)

Tipos de variables:
TARGET       int64
LOAN         int64
MORTDUE    float64
VALUE      float64
YOJ        float64
DEROG      float64
DELINQ     float64
CLAGE      float64
NINQ       float64
CLNO       float64
DEBTINC    float64
dtype: object


In [ ]:
print("Valores faltantes por columna:")
print(dataset.isna().sum())

Valores faltantes por columna:
TARGET     0
LOAN       0
MORTDUE    0
VALUE      0
YOJ        0
DEROG      0
DELINQ     0
CLAGE      0
NINQ       0
CLNO       0
DEBTINC    0
dtype: int64


In [ ]:
print("Distribución de TARGET en cantidad:")
print(dataset["TARGET"].value_counts())
print()
print("Distribución de TARGET en proporción:")
print(dataset["TARGET"].value_counts(normalize=True))

Distribución de TARGET en cantidad:
TARGET
0    1545
1     309
Name: count, dtype: int64

Distribución de TARGET en proporción:
TARGET
0   0.8333
1   0.1667
Name: proportion, dtype: float64


In [ ]:
dataset.describe()

,TARGET,LOAN,MORTDUE,VALUE,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC
count,1854.0000,1854.0000,1854.0000,1854.0000,1854.0000,1854.0000,1854.0000,1854.0000,1854.0000,1854.0000,1854.0000
mean,0.1667,19111.7584,76316.0518,107321.0885,8.9002,0.1877,0.3198,180.3008,1.1289,21.8571,34.5734
std,0.3728,11000.3460,46227.0266,56039.6851,7.5527,0.7049,0.9285,84.8383,1.6646,9.5108,9.3088
min,0.0000,1700.0000,5627.0000,21144.0000,0.0000,0.0000,0.0000,0.4867,0.0000,0.0000,0.8381
25%,0.0000,12000.0000,48984.7500,70787.2500,3.0000,0.0000,0.0000,116.9707,0.0000,16.0000,29.4272
50%,0.0000,17000.0000,67201.0000,94198.0000,7.0000,0.0000,0.0000,174.9678,1.0000,21.0000,35.3634
75%,0.0000,23900.0000,93731.5000,122976.2500,13.0000,0.0000,0.0000,232.2618,2.0000,27.0000,39.3580
max,1.0000,89800.0000,399412.0000,512650.0000,41.0000,10.0000,10.0000,1168.2336,13.0000,65.0000,144.1890


In [ ]:
# Diferencias promedio entre las clases
# Esto ayuda a pensar qué variables podrían ser importantes
dataset.groupby("TARGET").mean().T

TARGET,0,1
LOAN,19319.5469,18072.8155
MORTDUE,76798.1178,73905.7217
VALUE,108209.4511,102879.2751
YOJ,9.1670,7.5663
DEROG,0.0990,0.6311
DELINQ,0.1877,0.9806
CLAGE,186.3186,150.2116
NINQ,1.0259,1.6440
CLNO,21.7502,22.3916
DEBTINC,33.4635,40.1230


In [ ]:
# Correlación simple de cada variable con TARGET.
dataset.corr(numeric_only=True)["TARGET"].sort_values(ascending=False)

,TARGET
TARGET,1.0000
DELINQ,0.3183
DEROG,0.2814
DEBTINC,0.2667
NINQ,0.1384
CLNO,0.0251
MORTDUE,-0.0233
VALUE,-0.0355
LOAN,-0.0422
YOJ,-0.0790



Documentación:

- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

## Ejercicio 1: Descripción de los datos y la tarea

**1. ¿De qué se trata el conjunto de datos?**

El conjunto de datos contiene información de préstamos/créditos con garantía hipotecaria. El problema está relacionado con un banco que quiere construir un modelo para apoyar el proceso de decisión sobre solicitudes de crédito.

**2. ¿Cuál es la variable objetivo que hay que predecir? ¿Qué significado tiene?**

La variable objetivo es `TARGET`. Es una variable binaria:

- `TARGET = 0`: el préstamo fue pagado.
- `TARGET = 1`: el cliente incumplió el préstamo o tuvo una mora grave.

Por lo tanto, el modelo intenta predecir si un cliente será riesgoso o no.

**3. ¿Qué información está disponible para hacer la predicción?**

Las variables predictoras son:

- `LOAN`: monto del préstamo solicitado.
- `MORTDUE`: monto adeudado de hipoteca existente.
- `VALUE`: valor de la propiedad.
- `YOJ`: años en el empleo actual.
- `DEROG`: cantidad de reportes negativos importantes.
- `DELINQ`: cantidad de líneas de crédito con mora.
- `CLAGE`: antigüedad de la línea de crédito más antigua, en meses.
- `NINQ`: cantidad de consultas de crédito recientes.
- `CLNO`: cantidad de líneas de crédito.
- `DEBTINC`: relación deuda/ingreso.

**4. ¿Qué atributos parecen más determinantes?**

A priori, las variables más importantes podrían ser `DEBTINC`, `DELINQ`, `DEROG`, `NINQ` y `CLAGE`. Esto tiene sentido porque están relacionadas con endeudamiento, antecedentes de mora, reportes negativos y comportamiento crediticio previo.

En el análisis exploratorio también se ve que `TARGET` está desbalanceada: la clase `0` representa aproximadamente el 83,3% de los casos y la clase `1` aproximadamente el 16,7%. Por eso la accuracy sola puede ser engañosa.

## Preguntas orientadoras para el análisis del dataset

Antes de entrenar cualquier modelo, respondamos con texto:

1. ¿De qué se trata este conjunto de datos?

   Este conjunto de datos contiene información sobre préstamos otorgados por un banco o entidad financiera. Cada fila representa un préstamo o solicitud de préstamo, y las columnas describen características del cliente, del préstamo y de su historial crediticio.

2. ¿Qué representa la variable `TARGET`?

   La variable TARGET es la variable objetivo que queremos predecir. En este caso, indica si el préstamo fue pagado correctamente o si presentó incumplimiento.

3. ¿Qué significa la clase `0` y qué significa la clase `1`?

   La clase 0 representa a los clientes que pagaron el préstamo, es decir, casos sin incumplimiento. La clase 1 representa a los clientes que no pagaron correctamente o que tuvieron un incumplimiento grave.

4. ¿Cuál es el problema que intenta resolver el banco?

     El problema que intenta resolver el banco es anticipar qué clientes tienen mayor riesgo de no pagar el préstamo. Esto le permitiría tomar mejores decisiones al momento de aprobar o rechazar un crédito, ajustar condiciones del préstamo o realizar un seguimiento más cercano de ciertos clientes.

5. ¿Qué variables predictoras tenemos disponibles?

     Las variables predictoras disponibles son aquellas que describen el préstamo y la situación financiera o crediticia del cliente. Entre ellas se encuentran variables como el monto del préstamo, el valor de la propiedad, deuda hipotecaria, antigüedad laboral, cantidad de líneas de crédito, historial de mora, consultas recientes, relación deuda-ingreso, motivo del préstamo y tipo de trabajo.

6. ¿Qué variables creemos que podrían ser más importantes?

     Algunas variables que podrían ser más importantes son aquellas relacionadas con el comportamiento crediticio del cliente, como DEBTINC, DELINQ, DEROG, CLAGE y NINQ. También podrían ser relevantes variables como LOAN, VALUE y MORTDUE, porque están relacionadas con el monto del préstamo y la garantía disponible.


7. ¿Hay variables que podrían estar relacionadas entre sí?

     Es posible que algunas variables estén relacionadas entre sí. Por ejemplo, MORTDUE y VALUE podrían estar vinculadas porque ambas se relacionan con la propiedad. También DELINQ, DEROG y NINQ pueden estar relacionadas con el historial crediticio del cliente. Además, LOAN podría relacionarse con DEBTINC, ya que un préstamo más alto puede afectar la relación entre deuda e ingresos.

8. ¿Qué información adicional nos gustaría tener para comprender mejor el problema?

     Para comprender mejor el problema, sería útil contar con información adicional, como ingresos mensuales del cliente, tasa de interés del préstamo, plazo de pago, monto de la cuota, edad del cliente, zona geográfica, historial crediticio más completo y fecha en la que fue otorgado el préstamo. Estos datos permitirían analizar mejor el riesgo crediticio y mejorar la capacidad predictiva del modelo.



## División entre variables predictoras y variable objetivo

Usamos `TARGET` como etiqueta y el resto de las columnas como variables de entrada.

Además usamos `stratify=y` para que entrenamiento y test mantengan una proporción similar de clases.

In [ ]:
X = dataset.iloc[:, 1:]
y = dataset["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print()
print("Proporción de clases en entrenamiento:")
print(y_train.value_counts(normalize=True))
print()
print("Proporción de clases en test:")
print(y_test.value_counts(normalize=True))

X_train: (1483, 10)
X_test : (371, 10)

Proporción de clases en entrenamiento:
TARGET
0   0.8334
1   0.1666
Name: proportion, dtype: float64

Proporción de clases en test:
TARGET
0   0.8329
1   0.1671
Name: proportion, dtype: float64


## Función auxiliar para evaluar modelos

Como tenemos que reportar las mismas métricas varias veces, armamos una función simple.

Las métricas usadas son:

- accuracy;
- precision;
- recall;
- F1-score;
- matriz de confusión.

La clase positiva es `TARGET = 1`, es decir, clientes que incumplen o tienen mora grave.

In [ ]:
def evaluar_modelo(modelo, X_train_eval, X_test_eval, y_train, y_test, nombre_modelo):
    filas = []

    for nombre_conjunto, X_eval, y_real in [
        ("train", X_train_eval, y_train),
        ("test", X_test_eval, y_test)
    ]:
        y_pred = modelo.predict(X_eval)

        acc = accuracy_score(y_real, y_pred)
        prec = precision_score(y_real, y_pred, zero_division=0)
        rec = recall_score(y_real, y_pred, zero_division=0)
        f1 = f1_score(y_real, y_pred, zero_division=0)
        cm = confusion_matrix(y_real, y_pred)

        filas.append({
            "modelo": nombre_modelo,
            "conjunto": nombre_conjunto,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1
        })

        print("=" * 70)
        print(nombre_modelo, "-", nombre_conjunto)
        print("Accuracy :", round(acc, 4))
        print("Precision:", round(prec, 4))
        print("Recall   :", round(rec, 4))
        print("F1-score :", round(f1, 4))
        print()
        print("Matriz de confusión:")
        print(cm)
        print()
        print("Classification report:")
        print(classification_report(y_real, y_pred, zero_division=0))

    return pd.DataFrame(filas)


# Modelo lineal con SGDClassifier

En esta parte vamos a usar un clasificador lineal entrenado mediante descenso de gradiente estocástico.

Antes de evaluar resultados, respondamos:

1. ¿Qué significa que sea un modelo lineal?
2. ¿Qué función de pérdida estamos usando?
3. ¿Qué hiperparámetros aparecen en la documentación?
4. ¿Qué valores toma el modelo por defecto?
5. ¿Por qué la tasa de aprendizaje y la regularización pueden afectar el resultado?

Esto conecta directamente con la clase teórica sobre función de costo, optimización y descenso de gradiente.


Un modelo lineal separa las clases usando una combinación lineal de las variables. En este caso usamos `SGDClassifier`, que entrena mediante descenso de gradiente estocástico.

Como las variables tienen escalas muy distintas, antes de aplicar SGD escalamos las variables con `StandardScaler`. Esto no cambia el modelo elegido; solamente evita que variables con valores muy grandes dominen el proceso de entrenamiento.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Ejercicio 2: Predicción con Modelos Lineales

En este ejercicio se entrenarán modelos lineales de clasificación para predecir la variable objetivo.

Para ello, deberán utilizar la clase SGDClassifier de scikit-learn.

Documentación:
- https://scikit-learn.org/stable/modules/sgd.html
- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html


### Ejercicio 2.1: SGDClassifier con hiperparámetros por defecto

Entrenar y evaluar el clasificador SGDClassifier usando los valores por omisión de scikit-learn para todos los parámetros. Únicamente **fijar la semilla aleatoria** para hacer repetible el experimento.

Evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión

Entrenamos el modelo con los valores por defecto y fijamos `random_state=0`

In [ ]:
sgd_default = SGDClassifier(random_state=0)
sgd_default.fit(X_train_scaled, y_train)

resultados_sgd_default = evaluar_modelo(
    sgd_default,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    "SGDClassifier default"
)

resultados_sgd_default


SGDClassifier default - train
Accuracy : 0.851
Precision: 0.63
Recall   : 0.2551
F1-score : 0.3631

Matriz de confusión:
[[1199   37]
 [ 184   63]]

Classification report:
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1236
           1       0.63      0.26      0.36       247

    accuracy                           0.85      1483
   macro avg       0.75      0.61      0.64      1483
weighted avg       0.83      0.85      0.82      1483

SGDClassifier default - test
Accuracy : 0.8544
Precision: 0.6538
Recall   : 0.2742
F1-score : 0.3864

Matriz de confusión:
[[300   9]
 [ 45  17]]

Classification report:
              precision    recall  f1-score   support

           0       0.87      0.97      0.92       309
           1       0.65      0.27      0.39        62

    accuracy                           0.85       371
   macro avg       0.76      0.62      0.65       371
weighted avg       0.83      0.85      0.83       371



,modelo,conjunto,accuracy,precision,recall,f1
0,SGDClassifier default,train,0.8510,0.6300,0.2551,0.3631
1,SGDClassifier default,test,0.8544,0.6538,0.2742,0.3864


Interpretación del modelo SGD por defecto

El modelo obtiene una accuracy relativamente alta porque la mayoría de los casos pertenecen a la clase 0. Sin embargo, el recall de la clase 1 es bajo. Esto significa que detecta pocos clientes que efectivamente incumplen.

En este problema eso es importante, porque si un cliente riesgoso queda clasificado como no riesgoso, el banco podría aprobar un crédito que luego no se pague.

# Búsqueda de hiperparámetros con validación cruzada

Ahora no queremos quedarnos con una sola configuración del modelo.

Vamos a probar varias combinaciones de hiperparámetros y evaluarlas mediante validación cruzada.

Cuando analicemos los resultados de `GridSearchCV`, no miremos solamente el mejor score. También observemos:

1. `best_params_`: mejor combinación de hiperparámetros.
2. `best_score_`: mejor desempeño promedio en validación cruzada.
3. `mean_test_score`: media del score en validación.
4. `std_test_score`: variabilidad entre folds.
5. `mean_train_score`: desempeño promedio en entrenamiento, si está disponible.

Una configuración con media alta y desviación estándar baja suele ser más estable que una configuración con media alta pero gran variabilidad.


### Ejercicio 2.2: Ajuste de Hiperparámetros

Seleccionar valores para los hiperparámetros principales del SGDClassifier. Como mínimo, probar diferentes funciones de loss, tasas de entrenamiento y tasas de regularización.

Para ello, usar grid-search y 5-fold cross-validation sobre el conjunto de entrenamiento para explorar muchas combinaciones posibles de valores.

Reportar accuracy promedio y varianza para todas las configuraciones.

Para la mejor configuración encontrada, evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión

Documentación:
- https://scikit-learn.org/stable/modules/grid_search.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [ ]:
param_grid_sgd = {
    "loss": ["hinge", "log_loss", "modified_huber", "perceptron"],
    "alpha": [0.0001, 0.001, 0.01, 0.1],
    "learning_rate": ["constant", "optimal", "adaptive"],
    "eta0": [0.001, 0.01, 0.1]
}

sgd_grid = GridSearchCV(
    SGDClassifier(random_state=0, max_iter=2000, tol=1e-3),
    param_grid_sgd,
    scoring="accuracy",
    cv=5,
    return_train_score=True
)

sgd_grid.fit(X_train_scaled, y_train)

print("Mejores hiperparámetros:")
print(sgd_grid.best_params_)
print()
print("Mejor accuracy promedio en validación cruzada:")
print(round(sgd_grid.best_score_, 4))


Mejores hiperparámetros:
{'alpha': 0.0001, 'eta0': 0.01, 'learning_rate': 'constant', 'loss': 'hinge'}

Mejor accuracy promedio en validación cruzada:
0.8759


In [ ]:
resultados_grid_sgd = pd.DataFrame(sgd_grid.cv_results_)

resultados_grid_sgd = resultados_grid_sgd[[
    "param_loss",
    "param_alpha",
    "param_learning_rate",
    "param_eta0",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "rank_test_score"
]].copy()

resultados_grid_sgd["var_test_score"] = resultados_grid_sgd["std_test_score"] ** 2

resultados_grid_sgd = resultados_grid_sgd.sort_values("rank_test_score")
resultados_grid_sgd

,param_loss,param_alpha,param_learning_rate,param_eta0,mean_test_score,std_test_score,mean_train_score,rank_test_score,var_test_score
12,hinge,0.0001,constant,0.0100,0.8759,0.0090,0.8729,1,0.0001
48,hinge,0.0010,constant,0.0100,0.8753,0.0100,0.8727,2,0.0001
49,log_loss,0.0010,constant,0.0100,0.8726,0.0125,0.8734,3,0.0002
13,log_loss,0.0001,constant,0.0100,0.8726,0.0125,0.8732,3,0.0002
32,hinge,0.0001,adaptive,0.1000,0.8719,0.0101,0.8719,5,0.0001
68,hinge,0.0010,adaptive,0.1000,0.8719,0.0101,0.8719,5,0.0001
20,hinge,0.0001,adaptive,0.0100,0.8712,0.0088,0.8721,7,0.0001
33,log_loss,0.0001,adaptive,0.1000,0.8712,0.0117,0.8732,7,0.0001
57,log_loss,0.0010,adaptive,0.0100,0.8712,0.0117,0.8729,7,0.0001
56,hinge,0.0010,adaptive,0.0100,0.8712,0.0088,0.8719,7,0.0001


In [ ]:
mejor_sgd = sgd_grid.best_estimator_

resultados_sgd_ajustado = evaluar_modelo(
    mejor_sgd,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test,
    "SGDClassifier ajustado"
)

resultados_sgd_ajustado

SGDClassifier ajustado - train
Accuracy : 0.8732
Precision: 0.8933
Recall   : 0.2713
F1-score : 0.4161

Matriz de confusión:
[[1228    8]
 [ 180   67]]

Classification report:
              precision    recall  f1-score   support

           0       0.87      0.99      0.93      1236
           1       0.89      0.27      0.42       247

    accuracy                           0.87      1483
   macro avg       0.88      0.63      0.67      1483
weighted avg       0.88      0.87      0.84      1483

SGDClassifier ajustado - test
Accuracy : 0.8841
Precision: 1.0
Recall   : 0.3065
F1-score : 0.4691

Matriz de confusión:
[[309   0]
 [ 43  19]]

Classification report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93       309
           1       1.00      0.31      0.47        62

    accuracy                           0.88       371
   macro avg       0.94      0.65      0.70       371
weighted avg       0.90      0.88      0.86       371



,modelo,conjunto,accuracy,precision,recall,f1
0,SGDClassifier ajustado,train,0.8732,0.8933,0.2713,0.4161
1,SGDClassifier ajustado,test,0.8841,1.0000,0.3065,0.4691


Interpretación del ajuste de SGD

El ajuste de hiperparámetros mejora la accuracy respecto del SGD por defecto. De todos modos, el recall para la clase 1 sigue siendo bajo. Es decir, aunque el modelo casi no marca como riesgoso a clientes que en realidad pagan, todavía deja pasar varios clientes que sí incumplen.

# Árbol de decisión

Ahora vamos a repetir el análisis usando un árbol de decisión.

Este modelo tiene una interpretación diferente al clasificador lineal:

- divide el espacio de atributos mediante reglas;
- puede capturar relaciones no lineales;
- puede sobreajustar si crece demasiado.

Preguntas orientadoras:

1. ¿Qué profundidad alcanza el árbol por defecto?
2. ¿Hay evidencia de sobreajuste?
3. ¿Qué hiperparámetros podemos ajustar?
4. ¿Qué criterio conviene probar: `gini`, `entropy` o `log_loss`?
5. ¿Qué efecto tiene `max_depth`?
6. ¿Qué efecto tiene `min_samples_leaf`?


## Ejercicio 3: Árboles de Decisión

En este ejercicio se entrenarán árboles de decisión para predecir la variable objetivo.

Para ello, deberán utilizar la clase DecisionTreeClassifier de scikit-learn.

Documentación:
- https://scikit-learn.org/stable/modules/tree.html
  - https://scikit-learn.org/stable/modules/tree.html#tips-on-practical-use
- https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
- https://scikit-learn.org/stable/auto_examples/tree/plot_unveil_tree_structure.html

### Ejercicio 3.1: DecisionTreeClassifier con hiperparámetros por defecto

Entrenar y evaluar el clasificador DecisionTreeClassifier usando los valores por omisión de scikit-learn para todos los parámetros. Únicamente **fijar la semilla aleatoria** para hacer repetible el experimento.

Evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión


In [ ]:
tree_default = DecisionTreeClassifier(random_state=0)
tree_default.fit(X_train, y_train)

print("Profundidad del árbol:", tree_default.get_depth())
print("Cantidad de hojas:", tree_default.get_n_leaves())

Profundidad del árbol: 20
Cantidad de hojas: 147


In [ ]:
resultados_tree_default = evaluar_modelo(
    tree_default,
    X_train,
    X_test,
    y_train,
    y_test,
    "DecisionTree default"
)

resultados_tree_default

DecisionTree default - train
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0

Matriz de confusión:
[[1236    0]
 [   0  247]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1236
           1       1.00      1.00      1.00       247

    accuracy                           1.00      1483
   macro avg       1.00      1.00      1.00      1483
weighted avg       1.00      1.00      1.00      1483

DecisionTree default - test
Accuracy : 0.8787
Precision: 0.6545
Recall   : 0.5806
F1-score : 0.6154

Matriz de confusión:
[[290  19]
 [ 26  36]]

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.94      0.93       309
           1       0.65      0.58      0.62        62

    accuracy                           0.88       371
   macro avg       0.79      0.76      0.77       371
weighted avg       0.87      0.88      0.88       371



,modelo,conjunto,accuracy,precision,recall,f1
0,DecisionTree default,train,1.0000,1.0000,1.0000,1.0000
1,DecisionTree default,test,0.8787,0.6545,0.5806,0.6154


Interpretación del árbol por defecto

El árbol por defecto obtiene accuracy de entrenamiento igual a 1. Esto indica una señal clara de sobreajuste: el modelo aprendió perfectamente los datos de entrenamiento.

En test el rendimiento baja. Aun así, el recall de la clase 1 es mayor que el de los modelos lineales anteriores, por lo que detecta más casos de incumplimiento. El problema es que esa mejora viene acompañada de un árbol muy profundo y menos estable.

### Ejercicio 3.2: Ajuste de Hiperparámetros

Seleccionar valores para los hiperparámetros principales del DecisionTreeClassifier. Como mínimo, probar diferentes criterios de partición (criterion), profundidad máxima del árbol (max_depth), y cantidad mínima de samples por hoja (min_samples_leaf).

Para ello, usar grid-search y 5-fold cross-validation sobre el conjunto de entrenamiento para explorar muchas combinaciones posibles de valores.

Reportar accuracy promedio y varianza para todas las configuraciones.

Para la mejor configuración encontrada, evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión


Documentación:
- https://scikit-learn.org/stable/modules/grid_search.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [ ]:
param_grid_tree = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "min_samples_leaf": [1, 5, 10, 20, 30]
}

tree_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_grid_tree,
    scoring="accuracy",
    cv=5,
    return_train_score=True
)

tree_grid.fit(X_train, y_train)

print("Mejores hiperparámetros:")
print(tree_grid.best_params_)
print()
print("Mejor accuracy promedio en validación cruzada:")
print(round(tree_grid.best_score_, 4))

Mejores hiperparámetros:
{'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 10}

Mejor accuracy promedio en validación cruzada:
0.884


In [ ]:
resultados_grid_tree = pd.DataFrame(tree_grid.cv_results_)

resultados_grid_tree = resultados_grid_tree[[
    "param_criterion",
    "param_max_depth",
    "param_min_samples_leaf",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "rank_test_score"
]].copy()

resultados_grid_tree["var_test_score"] = resultados_grid_tree["std_test_score"] ** 2

resultados_grid_tree = resultados_grid_tree.sort_values("rank_test_score")
resultados_grid_tree

,param_criterion,param_max_depth,param_min_samples_leaf,mean_test_score,std_test_score,mean_train_score,rank_test_score,var_test_score
52,entropy,4,10,0.8840,0.0131,0.8887,1,0.0002
92,log_loss,4,10,0.8840,0.0131,0.8887,1,0.0002
65,entropy,8,1,0.8820,0.0093,0.9370,3,0.0001
105,log_loss,8,1,0.8820,0.0093,0.9370,3,0.0001
90,log_loss,4,1,0.8813,0.0143,0.8891,5,0.0002
50,entropy,4,1,0.8813,0.0143,0.8891,5,0.0002
51,entropy,4,5,0.8813,0.0122,0.8884,7,0.0001
91,log_loss,4,5,0.8813,0.0122,0.8884,7,0.0001
70,entropy,10,1,0.8813,0.0083,0.9543,9,0.0001
110,log_loss,10,1,0.8813,0.0083,0.9543,9,0.0001


In [ ]:
mejor_tree = tree_grid.best_estimator_

print("Profundidad del mejor árbol:", mejor_tree.get_depth())
print("Cantidad de hojas del mejor árbol:", mejor_tree.get_n_leaves())

Profundidad del mejor árbol: 4
Cantidad de hojas del mejor árbol: 10


In [ ]:
resultados_tree_ajustado = evaluar_modelo(
    mejor_tree,
    X_train,
    X_test,
    y_train,
    y_test,
    "DecisionTree ajustado"
)

resultados_tree_ajustado

DecisionTree ajustado - train
Accuracy : 0.8908
Precision: 0.9293
Recall   : 0.3725
F1-score : 0.5318

Matriz de confusión:
[[1229    7]
 [ 155   92]]

Classification report:
              precision    recall  f1-score   support

           0       0.89      0.99      0.94      1236
           1       0.93      0.37      0.53       247

    accuracy                           0.89      1483
   macro avg       0.91      0.68      0.73      1483
weighted avg       0.89      0.89      0.87      1483

DecisionTree ajustado - test
Accuracy : 0.8868
Precision: 0.9545
Recall   : 0.3387
F1-score : 0.5

Matriz de confusión:
[[308   1]
 [ 41  21]]

Classification report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94       309
           1       0.95      0.34      0.50        62

    accuracy                           0.89       371
   macro avg       0.92      0.67      0.72       371
weighted avg       0.89      0.89      0.86       371



,modelo,conjunto,accuracy,precision,recall,f1
0,DecisionTree ajustado,train,0.8908,0.9293,0.3725,0.5318
1,DecisionTree ajustado,test,0.8868,0.9545,0.3387,0.5000


Interpretación del árbol ajustado

El árbol ajustado tiene menor complejidad que el árbol por defecto. La diferencia entre entrenamiento y test es mucho menor, por lo que hay menos señales de sobreajuste.

El modelo conserva una accuracy alta y una precision alta para la clase 1, aunque el recall sigue siendo bajo. Esto significa que cuando predice incumplimiento suele acertar, pero todavía no detecta todos los casos de incumplimiento.

# Conclusión final del trabajo práctico

Para cerrar el TP, escribamos una conclusión breve respondiendo:

1. ¿Cuál fue el mejor modelo encontrado?
2. ¿Con qué hiperparámetros?
3. ¿Qué métrica usamos para decidir?
4. ¿Por qué esa métrica es adecuada para este problema?
5. ¿El modelo parece estable entre folds?
6. ¿Hay señales de sobreajuste?
7. ¿Qué tipo de error nos preocupa más: falso positivo o falso negativo?
8. ¿Recomendaríamos usar este modelo en un contexto real? ¿Qué advertencias haríamos?

La respuesta no debería limitarse a copiar números. Debe justificar la decisión usando las métricas y el significado del problema.


# Sugerencia opcional: tabla comparativa final

Podemos resumir los resultados en una tabla como esta:

| Modelo | Mejor configuración | Accuracy test | Precision | Recall | F1 | Comentario |
|---|---|---:|---:|---:|---:|---|
| SGDClassifier | ... | ... | ... | ... | ... | ... |
| DecisionTreeClassifier | ... | ... | ... | ... | ... | ... |

Esta tabla ayuda a comparar modelos de manera ordenada.


In [ ]:
comparacion_final = pd.concat([
    resultados_sgd_default,
    resultados_sgd_ajustado,
    resultados_tree_default,
    resultados_tree_ajustado
], ignore_index=True)

comparacion_test = comparacion_final[comparacion_final["conjunto"] == "test"].copy()
comparacion_test.sort_values("accuracy", ascending=False)

,modelo,conjunto,accuracy,precision,recall,f1
7,DecisionTree ajustado,test,0.8868,0.9545,0.3387,0.5000
3,SGDClassifier ajustado,test,0.8841,1.0000,0.3065,0.4691
5,DecisionTree default,test,0.8787,0.6545,0.5806,0.6154
1,SGDClassifier default,test,0.8544,0.6538,0.2742,0.3864


## Conclusión final

El mejor modelo encontrado mediante validación cruzada y búsqueda de hiperparámetros fue el `DecisionTreeClassifier` ajustado, con:

- `criterion = "entropy"`;
- `max_depth = 4`;
- `min_samples_leaf = 10`.

Este modelo obtuvo la mejor accuracy en test entre los modelos ajustados y, además, no mostró el sobreajuste extremo del árbol por defecto. El árbol por defecto tenía accuracy perfecta en entrenamiento, lo cual indica que memorizó demasiado los datos.

Para decidir no conviene mirar solamente accuracy, porque la variable `TARGET` está desbalanceada: la mayoría de los casos son clase `0`. Por eso también hay que mirar precision, recall, F1 y matriz de confusión.

En este problema, la clase `1` representa incumplimiento o mora grave. Desde el punto de vista del banco, preocupa especialmente no detectar clientes que luego incumplen. Es decir, preocupa que el modelo prediga `0` cuando en realidad el valor era `1`. Por eso el recall de la clase `1` es importante.

La recomendación sería usar el árbol ajustado como modelo inicial, porque es más estable y más interpretable que el árbol sin restricciones. Sin embargo, en un contexto real no lo usaría todavía como decisión automática definitiva, porque el recall de la clase `1` sigue siendo bajo. Lo usaría como apoyo a la decisión y seguiría trabajando en mejorar la detección de clientes riesgosos.